GRU(Gated Recurrent Unit)

In [1]:
import pandas as pd

df=pd.read_csv('../datasets/nepali_sentiment.csv')
df.head()

,text,label
0,1 देशलाई आर्थिक राजनेतिक हिसाबले आत्मा निर्भर ...,"1,2"
1,1 मित्र्यु हामी सबैले दिन मर्नु,"0,2"
2,10 घण्टामा 500000 लाख view,2
3,100 खाकोई 70 करोड छाती थोकेर किनकी खाली पुस्ती...,"0,1,2"
4,100 रूपैयाँ बाड्दै ति दुखि दिन हरु k सूखी परिव...,"0,2"


In [2]:
df.isna().sum()

text     0
label    0
dtype: int64

In [3]:
df["label"].value_counts()

label
1        1279
0         922
0,1       514
1,2       401
2         357
0,2       356
0,1,2     177
Name: count, dtype: int64

In [4]:
df["text"] = df["text"].str.lower()

In [5]:
import re
def clean_text(text):
    text=re.sub(r"https?://\S+|www\.\S+"," ",text)
    text=re.sub(r"@\w+"," ",text)
    text=re.sub(r"#"," ",text)
    text=re.sub(r"&\w+"," ",text)
    text=re.sub(r"\d+"," ",text)
    text=re.sub(r"\s+"," ",text).strip()

    return text

In [6]:
df["text"]=df["text"].apply(clean_text)

In [7]:
from nltk.tokenize import word_tokenize

df["tokens"] = df["text"].apply(word_tokenize)

In [8]:
df["tokens"] = df["tokens"].apply(' '.join)

In [9]:
all_words = " ".join(df["tokens"]).split()

In [10]:
unique_words = sorted(list(set(all_words)))

In [11]:
vocab = {
    word: i + 1
    for i, word in enumerate(unique_words)
}

vocab_size = len(vocab) + 1

print("Vocabulary size:", vocab_size)

Vocabulary size: 13818


In [12]:
X_sequences = []

for sentence in df["tokens"]:

    seq = [
        vocab[word]
        for word in sentence.split()
        if word in vocab
    ]

    X_sequences.append(seq)

In [13]:
import numpy as np

max_len = 100

x_padded = np.zeros(
    (len(X_sequences), max_len),
    dtype=np.int32
)

for i, seq in enumerate(X_sequences):

    if len(seq) > 0:
        x_padded[i, :len(seq[:max_len])] = seq[:max_len]

print(x_padded)

[[ 6010  1586 10689 ...     0     0     0]
 [10040 13340 12080 ...     0     0     0]
 [ 3562 11080   791 ...     0     0     0]
 ...
 [ 2180  5211 12378 ...     0     0     0]
 [10048  9421  2713 ...     0     0     0]
 [ 9808  9625 10489 ...     0     0     0]]


In [ ]:
import torch
from torch.utils.data import DataLoader, TensorDataset

train_dataset = TensorDataset(
    torch.tensor(x_train, dtype=torch.long),
    torch.tensor(y_train, dtype=torch.long)
)

test_dataset = TensorDataset(
    torch.tensor(x_test, dtype=torch.long),
    torch.tensor(y_test, dtype=torch.long)
)

batch_size = 64

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False
)


In [ ]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print(f"Using device: {device}")


Using device: cpu


In [ ]:
import torch.nn as nn

class SimpleGRU(nn.Module):

    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim):

        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            embedding_dim,
            padding_idx=0
        )

        self.gru = nn.GRU(
            embedding_dim,
            hidden_dim,
            batch_first=True
        )

        self.fc = nn.Linear(
            hidden_dim,
            output_dim
        )

    def forward(self, text):

        embedded = self.embedding(text)

        out, hidden = self.gru(embedded)

        final_hidden = hidden[-1]

        return self.fc(final_hidden)

In [ ]:
EMBEDDING_DIM=100
HIDDEN_DIM=128
OUTPUT_DIM = num_class

model=SimpleGRU(vocab_size,EMBEDDING_DIM,HIDDEN_DIM,OUTPUT_DIM).to(device)
print(model)

SimpleGRU(
  (embedding): Embedding(13796, 100, padding_idx=0)
  (gru): GRU(100, 128, batch_first=True)
  (fc): Linear(in_features=128, out_features=7, bias=True)
)


In [ ]:
criterion=nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

In [ ]:
epochs=5

for epoch in range(epochs):
    model.train()
    
    epoch_loss = 0
    correct = 0
    total = 0
    
    for texts,labels in train_loader:
        texts = texts.to(device)
        labels = labels.to(device)
        
        optimizer.zero_grad()
        
        predictions=model(texts)
        
        loss=criterion(predictions,labels)
        
        loss.backward()
        
        optimizer.step()
        
        epoch_loss +=loss.item()
        _,predicted=torch.max(predictions.data,1)
        total+=labels.size(0)
        correct+=(predicted==labels).sum().item()
        
    accuracy=100*correct/total
    print(
        f"Epoch {epoch+1:02d}/{epochs:02d} | "
        f"Loss: {epoch_loss/len(train_loader):.4f} | "
        f"Train Accuracy: {accuracy:.2f}%"
    )

Epoch 01/05 | Loss: 1.8065 | Train Accuracy: 30.90%
Epoch 02/05 | Loss: 1.7746 | Train Accuracy: 31.96%
Epoch 03/05 | Loss: 1.7686 | Train Accuracy: 31.96%
Epoch 04/05 | Loss: 1.7830 | Train Accuracy: 31.96%
Epoch 05/05 | Loss: 1.7651 | Train Accuracy: 31.99%


bi direction -rnn, gru,lstm

In [ ]:
model.eval()
test_correct=0
test_total=0

with torch.no_grad():
    for texts,labels in test_loader:
        texts=texts.to(device)
        labels = labels.to(device)

        predictions = model(texts)
        _, predicted = torch.max(predictions.data, 1)
        test_total += labels.size(0)
        test_correct += (predicted == labels).sum().item()

test_accuracy = 100 * test_correct / test_total
print(f"\nFinal Test Accuracy: {test_accuracy:.2f}%")


Final Test Accuracy: 31.92%
